In [2]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import time
import csv
import os
import joblib
import winsound 
from plyer import notification
import pygetwindow as gw
from ultralytics import YOLO
from plyer import notification
import pygetwindow as gw
import threading
import pyttsx3

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
# State tracker to prevent overlapping voice tracks
voice_state = [False]
print("Voice Warning Engine initialized.")

Voice Warning Engine initialized.


In [4]:
# 1. Initialize Engines & Core Models
print("Booting Study Guardian Engines (Initializing YOLOv8 & MediaPipe)...")
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)

model = joblib.load('gaze_model.pkl')
object_model = YOLO("yolov8n.pt") # Fast nano model

# Initialize the state to track background voice warnings
voice_state = [False]

# --- OFFLINE VOICE WARNING ENGINE ---
def speak_warning_async(message):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 165) # Speaking speed
        engine.say(message)
        engine.runAndWait()
        engine.stop()
    except Exception as e:
        print(f"[VOICE ERROR]: {e}")
    finally:
        # Allow the system to accept speech triggers again
        voice_state[0] = False

# --- CORE APPLICATION SETTINGS ---
LOOK_AWAY_LIMIT = 3.0   
SLEEP_THRESHOLD = 2.0   
BREAK_TIME = 1 * 60     # Set to 1 minute as per your config
TARGET_STUDY_WINDOW = "Visual Studio Code" # Set to your IDE window title!

# --- SESSION ANALYTICS TRACKERS ---
start_time = time.time()
total_frames = 0
center_frames = 0
distracted_frames = 0
sleep_frames = 0
phone_frames = 0
downward_frames = 0     # Track time spent looking downward specifically

away_start_time = None
closed_start_time = None  
is_away = False
is_sleeping = False
phone_in_view = False   

# Sound alarm pacing tracker
last_alarm_time = 0

cap = cv2.VideoCapture(0)
print("\n>>> Guardian Engaged. Press 'q' to quit. <<<")

while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    current_time = time.time()
    elapsed = current_time - start_time
    total_frames += 1
    
    # --- DETECTOR A: MEDIAPIPE FACE GEOMETRY ---
    results = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    prediction = "center"  

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            left_iris = face_landmarks.landmark[468]
            left_outer = face_landmarks.landmark[33]
            left_inner = face_landmarks.landmark[133]
            right_outer = face_landmarks.landmark[263]
            nose = face_landmarks.landmark[1]        
            
            left_top_eyelid = face_landmarks.landmark[159]
            left_bottom_eyelid = face_landmarks.landmark[145]
            eye_height = left_bottom_eyelid.y - left_top_eyelid.y
            
            left_edge = face_landmarks.landmark[234]  
            right_edge = face_landmarks.landmark[454] 
            
            # Boundary tracking landmarks for vertical head-tilt orientation
            top_head = face_landmarks.landmark[10]    # Top edge of forehead
            bottom_chin = face_landmarks.landmark[152] # Bottom edge of chin

            eye_width = left_inner.x - left_outer.x
            iris_pos = left_iris.x - left_outer.x
            face_width = right_edge.x - left_edge.x
            nose_relative_pos = (nose.x - left_edge.x) / face_width
            
            dist_to_left = abs(left_outer.x - nose.x)
            dist_to_right = abs(right_outer.x - nose.x)
            face_symmetry = dist_to_left / (dist_to_right + 1e-6)

            # Mathematical head pitch calculation to catch looking down away from keyboard
            face_total_height = bottom_chin.y - top_head.y
            nose_height_ratio = (nose.y - top_head.y) / (face_total_height + 1e-6)

            if eye_height < 0.008:  
                prediction = "closed"
            elif nose_height_ratio > 0.68:  # Nose drops low relative to face bounds when looking down
                prediction = "downward"
            else:
                features_df = pd.DataFrame(
                    [[iris_pos, eye_width, nose_relative_pos, face_symmetry]], 
                    columns=['iris_pos', 'eye_width', 'nose_relative_pos', 'face_symmetry']
                )
                raw_prediction = model.predict(features_df)[0]
                probabilities = model.predict_proba(features_df)[0]
                class_index = list(model.classes_).index(raw_prediction)
                confidence = probabilities[class_index]
                
                if raw_prediction in ["left", "right"] and confidence >= 0.75:
                    prediction = raw_prediction
                else:
                    prediction = "center"

    # --- DETECTOR B: YOLO OBJECT DETECTION ---
    if total_frames % 5 == 0:
        phone_in_view = False
        obj_results = object_model(frame, verbose=False)[0]
        for box in obj_results.boxes:
            class_id = int(box.cls[0])
            class_name = object_model.names[class_id]
            if class_name in ["cell phone"]:
                phone_in_view = True

    # --- STATISTICAL REPORT LOGGER ---
    if prediction == "center" and not phone_in_view: center_frames += 1
    elif prediction == "closed": sleep_frames += 1
    elif prediction in ["left", "right"]: distracted_frames += 1
    elif prediction == "downward": downward_frames += 1
    if phone_in_view: phone_frames += 1

    # --- INTERVENTION ENGINE & SYSTEM AUTOMATION ---
    
    # 1. DROWSINESS EVALUATION ENGINE
    if prediction == "closed":
        if closed_start_time is None: closed_start_time = current_time
        if (current_time - closed_start_time) >= SLEEP_THRESHOLD:
            if not is_sleeping:
                is_sleeping = True
                if not voice_state[0]:
                    voice_state[0] = True
                    sleep_phrase = "Wake up! Don't let sleep ruin your study goals today."
                    print(f"\n[VOICE TRIGGERED]: {sleep_phrase}")
                    t = threading.Thread(target=speak_warning_async, args=(sleep_phrase,))
                    t.start()
            
            # Continuous persistent buzzing alert while eyes remain closed after speech
            if current_time - last_alarm_time >= 1.0:
                winsound.Beep(800, 200)  # Continuous Drowsy Beep Tone
                last_alarm_time = current_time
    else:
        closed_start_time = None
        is_sleeping = False

    # 2. SCREEN DISTRACTION, PHONE SCROLLING & DOWNWARD LOOKING ENGINE
    if prediction in ["left", "right", "downward"] or phone_in_view:
        if away_start_time is None: away_start_time = current_time
        if (current_time - away_start_time) >= LOOK_AWAY_LIMIT:
            
            if not is_away:
                is_away = True
                if not voice_state[0]:
                    voice_state[0] = True
                    
                    # Choose correct voice path dynamically including downward-tilt
                    if phone_in_view:
                        warning_phrase = "Put your phone away. Stay disciplined and finish your session."
                    elif prediction == "downward":
                        warning_phrase = "Eyes up! Looking down at your lap won't pass your technical assessments."
                    else:
                        warning_phrase = "Focus on your workspace. Stop wandering around."
                        
                    print(f"\n[VOICE TRIGGERED]: {warning_phrase}")
                    t = threading.Thread(target=speak_warning_async, args=(warning_phrase,))
                    t.start()
            
            # Continuous persistent ringing loop if distraction continues after speaking
            if current_time - last_alarm_time >= 1.0:
                winsound.Beep(1200, 150) # High urgency alert sound wave
                last_alarm_time = current_time
            
            # WINDOW AUTOMATION: Refocus Workspace Window
            try:
                target_windows = gw.getWindowsWithTitle(TARGET_STUDY_WINDOW)
                if target_windows:
                    study_win = target_windows[0]
                    if not study_win.isActive:
                        if study_win.isMinimized: study_win.restore()
                        study_win.activate()
            except Exception:
                pass  
    else:
        away_start_time = None
        is_away = False

    if elapsed >= BREAK_TIME:
        notification.notify(
            title="🎯 Mission Accomplished! Session Complete.", 
            message="Your focus score is stellar! Step away from the screen, stretch, and get some fresh air. You've earned it!"
        )
        # Play a cheerful multi-tone victory sound instead of a standard boring beep
        winsound.Beep(440, 100)  
        winsound.Beep(554, 100)  
        winsound.Beep(659, 100)  
        winsound.Beep(880, 250)  
        start_time = current_time

    # --- GRAPHICAL UI LAYER ---
    status_alert = (is_away or is_away or (prediction == "downward") or phone_in_view)
    status_color = (0, 0, 255) if status_alert else (0, 255, 0)
    
    if prediction == "closed":
        display_gaze = "DROWSY"
    elif prediction == "downward":
        display_gaze = "LOOKING DOWNWARD"
    elif prediction in ["left", "right"]:
        display_gaze = "DISTRACTED"
    else:
        display_gaze = "CENTER"
    
    cv2.rectangle(frame, (0, 0), (w, 85), (0, 0, 0), -1)
    cv2.putText(frame, f"STATUS: {display_gaze}", (20, 35), 0, 0.7, status_color, 2)
    cv2.putText(frame, f"PHONE SCROLLING: {'YES' if phone_in_view else 'NO'}", (20, 65), 0, 0.6, (255, 255, 255), 1)
    cv2.putText(frame, f"TIMER: {int(elapsed//60)}m", (w-140, 45), 0, 0.7, (255, 255, 255), 2)

    cv2.imshow('Study Guardian Ultimate Workspace', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

# --- SUMMARY REPORT DISPLAY ---
print("\n" + "="*40)
print("       STUDY GUARDIAN REPORT CARD       ")
print("="*40)
tracking_duration = int(time.time() - start_time)
print(f"Total Tracking Window : {tracking_duration} seconds")
if total_frames > 0:
    focus_percentage = (center_frames / total_frames) * 100
    print(f"Overall Focus Score   : {focus_percentage:.1f}%")
    print(f"Time Spent Focused    : {int((center_frames/total_frames)*tracking_duration)}s")
    print(f"Time Spent Looking Dn : {int((downward_frames/total_frames)*tracking_duration)}s")
    print(f"Time Spent Distracted : {int((distracted_frames/total_frames)*tracking_duration)}s")
    print(f"Time Spent Drowsy     : {int((sleep_frames/total_frames)*tracking_duration)}s")
    print(f"Phone Exposure Time   : {int((phone_frames/total_frames)*tracking_duration)}s")
print("="*40)

Booting Study Guardian Engines (Initializing YOLOv8 & MediaPipe)...

>>> Guardian Engaged. Press 'q' to quit. <<<

[VOICE TRIGGERED]: Focus on your workspace. Stop wandering around.

[VOICE TRIGGERED]: Put your phone away. Stay disciplined and finish your session.

[VOICE TRIGGERED]: Wake up! Don't let sleep ruin your study goals today.

       STUDY GUARDIAN REPORT CARD       
Total Tracking Window : 2 seconds
Overall Focus Score   : 82.7%
Time Spent Focused    : 1s
Time Spent Looking Dn : 0s
Time Spent Distracted : 0s
Time Spent Drowsy     : 0s
Phone Exposure Time   : 0s
